# Week 6 — Multi-table Joins and Complex Aggregations: Three-Table Joins
## Phase 2b SQL | PORA Academy Cohort 7 — **Exercises**

Wednesday's demo did two things. It chained `order_items → products → product_category_translation` to put revenue next to an English category name, and it bridged `order_items` to `order_reviews` on `order_id` to ask whether price tracks satisfaction. It also gave you a warning to carry into every join you write from now on: `order_items`, `order_payments` and `order_reviews` all hold **more than one row per `order_id`**, so joining two of them and then aggregating quietly inflates the answer.

The four questions below put both halves to work. The first two extend the category chain — a new ranking, a new aggregate, a percentage that will truncate to zero if you forget `* 1.0`. The last two are the fan-out questions: each one joins two many-rows-per-order tables, so each one **must** pre-aggregate in a `WITH` CTE first and count with `COUNT(DISTINCT order_id)`. Q3 in particular re-runs the demo's own price-vs-review query the safe way, so you can see for yourself exactly how much fan-out was adding.

Each question comes as **three cells**:

1. A **question** with the task and an **Expected** result.
2. A blank `%%sql` answer cell — write your query where it says `-- Your query here`, capturing the result into a variable (e.g. `q1`).
3. A **check cell** (plain Python) — run it after your query. A ✅ means you got it right, and the cell then displays the table your query returned.

**Do not edit the check cells.** Run the setup cell first, then work top to bottom. The check cells read the exact column aliases each question asks for, so use the alias names as written, and always alias your tables (`oi`, `p`, `t`, `r`) — with three tables in play, a bare `product_category_name` is ambiguous and SQLite will refuse it.

🤖 **Using DeepSeek this week:** you may ask DeepSeek to draft these joins, but the prompt-then-verify protocol from the demo still applies. Tell it the tables, the join keys, the exact column aliases, and that the dialect is SQLite — then **run the query and check it against the Expected value before you trust it**. Joins are where AI-drafted SQL fails most dangerously: it will happily hand you a `COUNT(*)` over a fanned-out join, or drop the middle table, and neither mistake raises an error — they just return a plausible wrong number. The check cells are the verification step; never edit one to make a wrong query pass.

In [ ]:
# =====================================================================
# Olist SQL Setup — runs on BOTH Google Colab and a local machine.
# Run this cell FIRST. It loads the 8 Olist tables into a SQLite
# database and connects the %%sql magic to it. You should not need to
# edit anything unless auto-detection fails (see the two knobs below).
#
# Design notes:
# - We teach SQL with the %%sql cell magic (jupysql), not pd.read_sql().
# - jupysql opens its OWN connection, so the DB must be a real FILE
#   (a :memory: DB would be invisible to it).
# - We use jupysql (the maintained SQL magic). On Colab we install it,
#   because Colab ships the legacy ipython-sql, which (a) can't take a
#   connection by engine variable and (b) renders every result through
#   prettytable.__dict__[style], crashing on modern prettytable with
#   KeyError 'DEFAULT'/'SINGLE_BORDER'. jupysql fixes both.
# - autopandas=True makes every %%sql result a pandas DataFrame, which
#   lets the self-check cells assert on .iloc/.shape directly.
# =====================================================================
import os, glob, sqlite3, tempfile, zipfile
import pandas as pd

# --- Optional knobs (leave blank; only set if auto-detect fails) ------
LOCAL_DATA_DIR = ""   # local run: folder that holds olist_orders_dataset.csv
DRIVE_ZIP_PATH = ""   # Colab: full path to phase-2-python-sql.zip in your Drive
# ---------------------------------------------------------------------

# Detect Colab (google.colab only imports there). Outside Colab — including
# the content-pipeline validator — this falls through to the local branch.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ON_COLAB = True
except ModuleNotFoundError:
    ON_COLAB = False


def _colab_find_zip():
    """Locate phase-2-python-sql.zip in Drive WITHOUT a full recursive scan
    (globbing '/content/drive/MyDrive/**' walks the entire Drive over the
    network and can hang for many minutes). Try explicit paths first, then a
    depth- and count-bounded breadth-first search that prints progress."""
    if DRIVE_ZIP_PATH:
        if os.path.exists(DRIVE_ZIP_PATH):
            return DRIVE_ZIP_PATH
        raise FileNotFoundError(f"DRIVE_ZIP_PATH is set but not found: {DRIVE_ZIP_PATH}")

    target = "phase-2-python-sql.zip"
    # Fast, instant checks of the most likely spots (top of Drive + course folder).
    for cand in (
        f"/content/drive/MyDrive/{target}",
        f"/content/drive/MyDrive/Data Analysis and AI Automation Course Cohort 7/Dataset/{target}",
        f"/content/{target}",
    ):
        if os.path.exists(cand):
            return cand

    # Bounded BFS: depth <= 4, at most ~600 folders, skipping hidden dirs.
    print("Searching your Google Drive for phase-2-python-sql.zip ...")
    root, queue, scanned = "/content/drive/MyDrive", [("/content/drive/MyDrive", 0)], 0
    while queue:
        d, depth = queue.pop(0)
        hit = os.path.join(d, target)
        if os.path.exists(hit):
            return hit
        if depth >= 4:
            continue
        try:
            for e in os.scandir(d):
                if e.is_dir() and not e.name.startswith("."):
                    queue.append((e.path, depth + 1))
        except OSError:
            continue
        scanned += 1
        if scanned % 50 == 0:
            print(f"  ...scanned {scanned} folders")
        if scanned >= 600:
            break

    raise FileNotFoundError(
        "Could not quickly find phase-2-python-sql.zip in your Drive. Put the zip at the "
        "TOP of your Drive (My Drive) and re-run, or set DRIVE_ZIP_PATH at the top of this "
        "cell to its exact path.")


def _find_csv_dir():
    """Return the folder that actually contains olist_orders_dataset.csv."""
    roots = []
    env_dir = os.environ.get("OLIST_DATA_PATH", "")   # set by the pipeline validator
    if env_dir:
        roots.append(env_dir)
    if LOCAL_DATA_DIR:
        roots.append(LOCAL_DATA_DIR)

    if ON_COLAB:
        extract_path = "/content/olist_data"
        # unzip only the first time; reuse the extracted CSVs afterwards
        if not glob.glob(f"{extract_path}/**/olist_orders_dataset.csv", recursive=True):
            zip_path = _colab_find_zip()
            os.makedirs(extract_path, exist_ok=True)
            print(f"Unzipping {os.path.basename(zip_path)} ...")
            with zipfile.ZipFile(zip_path) as z:
                z.extractall(extract_path)
        roots.append(extract_path)
    else:
        # Local: search cwd (recursively) + a few common spots — never the whole
        # home dir (that recursive walk can be very slow). Set LOCAL_DATA_DIR if
        # your CSVs live elsewhere.
        roots += [os.getcwd(),
                  os.path.expanduser("~/Downloads"),
                  os.path.expanduser("~/Desktop"),
                  os.path.expanduser("~/olist")]

    for root in roots:
        if os.path.exists(os.path.join(root, "olist_orders_dataset.csv")):
            return root
        hits = glob.glob(os.path.join(root, "**", "olist_orders_dataset.csv"), recursive=True)
        if hits:
            return os.path.dirname(hits[0])

    raise FileNotFoundError(
        "Olist CSVs not found. Set LOCAL_DATA_DIR (local) or DRIVE_ZIP_PATH (Colab) at "
        "the top of this cell.")


DATA_DIR = _find_csv_dir()
print("Data folder:", DATA_DIR)

# Build a file-based SQLite DB shared by pandas (loading) and jupysql (querying).
DB_PATH = os.environ.get("OLIST_DB_PATH") or (
    "/content/olist.db" if ON_COLAB else os.path.join(tempfile.gettempdir(), "olist.db"))

tables = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "product_category_translation": "product_category_name_translation.csv",
}

conn = sqlite3.connect(DB_PATH)
for table_name, filename in tables.items():
    df = pd.read_csv(os.path.join(DATA_DIR, filename))
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    print(f"Loaded {table_name}: {len(df):,} rows")
conn.close()
print("\nDatabase ready.")

# On Colab, install jupysql so `%load_ext sql` loads it instead of the legacy
# ipython-sql (see header). Off Colab (local / pipeline validator) jupysql is
# already installed, so we skip the install and stay offline-safe.
if ON_COLAB:
    get_ipython().run_line_magic("pip", "install --quiet --upgrade jupysql")

get_ipython().run_line_magic("load_ext", "sql")

# Guard: if the legacy ipython-sql was already loaded earlier THIS session (e.g.
# an older cell ran first), the freshly installed jupysql cannot hot-swap in — a
# runtime restart is the only fix. jupysql exposes sql.connection.ConnectionManager;
# ipython-sql does not. Stop with a clear instruction instead of a later cryptic
# prettytable KeyError.
import sql.connection as _sqlconn
if not hasattr(_sqlconn, "ConnectionManager"):
    raise RuntimeError(
        "Legacy ipython-sql is active, not jupysql. On Colab: Runtime -> Restart session, "
        "then run THIS setup cell first (before any other cell). Locally: "
        "pip install --upgrade jupysql and restart the kernel."
    )

# Connect the %%sql magic to the SAME database file. autopandas=True is REQUIRED
# (see header). We connect with run_line_magic (not a literal `%sql` line) so the
# computed DB_PATH is interpolated correctly. Do NOT set SqlMagic.style.
get_ipython().run_line_magic("config", "SqlMagic.autopandas = True")
get_ipython().run_line_magic("config", "SqlMagic.feedback = 0")
get_ipython().run_line_magic("sql", f"sqlite:///{DB_PATH}")

# Verify (expected row counts — do not alter without re-running against data):
#   orders 99,441 | customers 99,441 | order_items 112,650 | order_payments 103,886
#   order_reviews 99,224 | products 32,951 | sellers 3,095 | product_category_translation 71


### Before you start — keep this table next to you

Every question below aggregates over a join, so before you write a single `SELECT`, remind yourself of each table's **grain** — what one row of it actually represents. You verified all four of these counts yourself in Wednesday's demo:

| Table | Rows | Distinct `order_id` | Grain |
|---|---|---|---|
| `orders` | 99,441 | 99,441 | one row per order — safe base |
| `order_items` | 112,650 | 98,666 | **many rows per order** (one per line item) |
| `order_payments` | 103,886 | 99,440 | **many rows per order** (instalments, vouchers) |
| `order_reviews` | 99,224 | 98,673 | **many rows per order** (547 orders reviewed twice) |

Two rules follow from it, and they decide whether your answers are right or merely plausible:

1. **Counting orders after a join is always `COUNT(DISTINCT order_id)`, never `COUNT(*)`.** All four questions need this. An order with three line items becomes three rows in the join; `COUNT(*)` would report it as three orders.
2. **Never `SUM`, `AVG` or `COUNT(*)` across a query that directly joins two or more of `order_items` / `order_payments` / `order_reviews`.** Collapse the extra table to one row per `order_id` in a `WITH` CTE *first*, then join that CTE. Questions 3 and 4 both need this; Questions 1 and 2 touch only `order_items`, so a plain chain is safe there.

Neither mistake raises an error. Both just hand you a number that is quietly too big.

## Question 1 — The cheap end of the catalogue

The demo ranked categories by **revenue** and found `health_beauty` on top. Your manager now wants the opposite view: the categories where the *typical item is cheapest*, because those are the ones that need volume to be worth stocking. Tiny niche categories would dominate such a list by accident, so she only wants categories with a real customer base behind them.

Using the same three-table chain as the demo (`order_items` → `products` → `product_category_translation`, joined on `product_id` then on `product_category_name`), return the **5 categories with the lowest average item price**, restricted to categories appearing in **at least 500 distinct orders**. Three columns:

- `category` — the English category name
- `order_count` — `COUNT(DISTINCT oi.order_id)`, not `COUNT(*)` (the join fans line items out)
- `avg_price` — `ROUND(AVG(oi.price), 2)`

Put the 500-order floor in a `HAVING` clause (it filters *groups*, so it cannot go in `WHERE`), then `ORDER BY avg_price ASC` and `LIMIT 5`.

**Expected:** 5 rows, starting with `electronics` — 2,550 orders at an average of R$57.91 — then `telephony` (R$71.21), `fashion_bags_accessories` (R$75.25), `books_general_interest` (R$84.73), and `furniture_decor` (R$87.56). Cross-check the last one against the demo's top-10 table: `furniture_decor` appears there too, at exactly R$87.56.

In [ ]:
%%sql q1 <<
-- Your query here

In [ ]:
# --- CHECK Q1 — do not edit ---
for col in ['category', 'order_count', 'avg_price']:
    assert col in q1.columns, f"Q1: missing the '{col}' column — check your SELECT aliases"
assert q1.shape[0] == 5, f"Q1: expected 5 rows, got {q1.shape[0]} — did you LIMIT 5?"
assert q1.iloc[0]['category'] == 'electronics', \
    (f"Q1: expected 'electronics' first, got '{q1.iloc[0]['category']}' — "
     f"order by avg_price ASC and keep the HAVING >= 500 floor")
assert int(q1.iloc[0]['order_count']) == 2550, \
    (f"Q1: expected electronics order_count = 2,550, got {int(q1.iloc[0]['order_count']):,} — "
     f"use COUNT(DISTINCT oi.order_id), not COUNT(*)")
assert abs(float(q1.iloc[0]['avg_price']) - 57.91) < 0.01, \
    f"Q1: expected electronics avg_price ≈ 57.91, got {q1.iloc[0]['avg_price']}"
assert list(q1['category']) == ['electronics', 'telephony', 'fashion_bags_accessories',
                               'books_general_interest', 'furniture_decor'], \
    f"Q1: unexpected category order — got {list(q1['category'])}"
assert abs(float(q1.iloc[4]['avg_price']) - 87.56) < 0.01, \
    f"Q1: expected furniture_decor avg_price ≈ 87.56, got {q1.iloc[4]['avg_price']}"
print("✅ Q1 correct")
q1  # show the result of your query

## Question 2 — How much of `bed_bath_table` is actually shipping?

`order_items` carries two money columns: `price` (what the product cost) and `freight_value` (what it cost to ship). The demo only ever summed `price`. Bulky categories can carry a freight bill that eats a serious share of what the customer pays, and `bed_bath_table` — duvets, pillows, towels — is a prime suspect: it had the *most* orders of any category (9,417) but only the third-highest revenue.

Take the demo's three-table chain, filter it to **`bed_bath_table` only** with a `WHERE` on `t.product_category_name_english`, and return one row with five columns:

- `category` — the English category name
- `order_count` — `COUNT(DISTINCT oi.order_id)`
- `total_price` — `ROUND(SUM(oi.price), 2)`
- `total_freight` — `ROUND(SUM(oi.freight_value), 2)`
- `freight_pct` — freight as a percentage of everything the customer paid, i.e. `SUM(freight_value) * 100.0 / (SUM(price) + SUM(freight_value))`, rounded to 2 decimals

Note the `100.0`. Write `100` instead and SQLite does integer division, truncating your percentage to `0` — the classic gotcha from Week 4, and it does not raise an error.

**Expected:** one row — `bed_bath_table`, 9,417 orders, R$1,036,988.68 in product revenue, R$204,693.04 in freight, and `freight_pct` ≈ 16.49. One real in every six the customer hands over for bed and bath goods is paying for the truck, not the towel.

In [ ]:
%%sql q2 <<
-- Your query here

In [ ]:
# --- CHECK Q2 — do not edit ---
for col in ['category', 'order_count', 'total_price', 'total_freight', 'freight_pct']:
    assert col in q2.columns, f"Q2: missing the '{col}' column — check your SELECT aliases"
assert q2.shape[0] == 1, f"Q2: expected a single row, got {q2.shape[0]} — filter to bed_bath_table"
assert q2.iloc[0]['category'] == 'bed_bath_table', \
    f"Q2: expected category 'bed_bath_table', got '{q2.iloc[0]['category']}'"
assert int(q2.iloc[0]['order_count']) == 9417, \
    (f"Q2: expected order_count = 9,417, got {int(q2.iloc[0]['order_count']):,} — "
     f"use COUNT(DISTINCT oi.order_id), not COUNT(*)")
assert abs(float(q2.iloc[0]['total_price']) - 1036988.68) < 0.01, \
    f"Q2: expected total_price ≈ 1,036,988.68, got {q2.iloc[0]['total_price']}"
assert abs(float(q2.iloc[0]['total_freight']) - 204693.04) < 0.01, \
    f"Q2: expected total_freight ≈ 204,693.04, got {q2.iloc[0]['total_freight']}"
got_pct = float(q2.iloc[0]['freight_pct'])
assert abs(got_pct - 16.49) < 0.01, \
    (f"Q2: expected freight_pct ≈ 16.49, got {got_pct} — a 0 here means integer division "
     f"truncated it; multiply by 100.0, not 100")
print("✅ Q2 correct")
q2  # show the result of your query

## Question 3 — Redo the demo's review query, this time without the fan-out

The demo's second query joined `order_items` straight to `order_reviews` and reported 10,854 orders worth R$1,812,828.22 at review score 1. Both tables hold many rows per `order_id`, so those numbers are inflated — the demo said as much in "Going deeper" and promised you'd see the size of the error. This is that question. You are also going to add something the demo could not measure: whether unhappy orders are *bigger* orders.

Write a query with **two CTEs**, each collapsing one fan-out table to exactly one row per `order_id`:

```sql
WITH items AS (
    SELECT order_id, COUNT(*) AS item_count, SUM(price) AS order_revenue
    FROM order_items
    GROUP BY order_id
),
rev AS (
    SELECT order_id, AVG(review_score) AS review_score
    FROM order_reviews
    GROUP BY order_id
)
```

Then `JOIN items i` to `rev r` on `order_id` — now a clean 1:1 link that cannot fan out — and group by `review_score` to return four columns:

- `review_score`
- `order_count` — `COUNT(DISTINCT i.order_id)`
- `avg_items_per_order` — `ROUND(AVG(i.item_count), 2)`
- `total_revenue` — `ROUND(SUM(i.order_revenue), 2)`

Add `WHERE r.review_score IN (1, 2, 3, 4, 5)` before the `GROUP BY`. That is not decoration: 123 orders were reviewed twice with *disagreeing* scores, so their average is something like 3.5 and belongs in no bucket at all. Finish with `ORDER BY review_score`.

**Expected:** 5 rows.

| review_score | order_count | avg_items_per_order | total_revenue |
|---|---|---|---|
| 1 | 10,778 | 1.31 | 1,799,630.54 |
| 2 | 3,063 | 1.25 | 446,014.60 |
| 3 | 8,083 | 1.16 | 1,031,697.47 |
| 4 | 18,986 | 1.11 | 2,515,304.79 |
| 5 | 56,885 | 1.11 | 7,660,968.12 |

Compare row 1 against the demo: 10,778 real orders, not 10,854, and R$1,799,630.54, not R$1,812,828.22 — fan-out had added 76 phantom orders and R$13,197.68 of phantom revenue to a single row. And look down the `avg_items_per_order` column: 1-star orders average 1.31 items while 5-star orders average 1.11. Bigger baskets go wrong more often — which is a genuinely useful finding, and one the fanned-out version of this query could not have produced.

In [ ]:
%%sql q3 <<
-- Your query here

In [ ]:
# --- CHECK Q3 — do not edit ---
for col in ['review_score', 'order_count', 'avg_items_per_order', 'total_revenue']:
    assert col in q3.columns, f"Q3: missing the '{col}' column — check your SELECT aliases"
assert q3.shape[0] == 5, \
    (f"Q3: expected 5 rows (one per score), got {q3.shape[0]} — keep "
     f"WHERE r.review_score IN (1, 2, 3, 4, 5)")
assert [int(round(float(s))) for s in q3['review_score']] == [1, 2, 3, 4, 5], \
    f"Q3: expected scores 1-5 in ascending order, got {list(q3['review_score'])}"
assert int(q3.iloc[0]['order_count']) == 10778, \
    (f"Q3: expected 10,778 orders at score 1, got {int(q3.iloc[0]['order_count']):,} — "
     f"10,854 means you joined order_items to order_reviews directly instead of "
     f"pre-aggregating each one in a CTE")
assert abs(float(q3.iloc[0]['avg_items_per_order']) - 1.31) < 0.01, \
    f"Q3: expected avg_items_per_order ≈ 1.31 at score 1, got {q3.iloc[0]['avg_items_per_order']}"
assert abs(float(q3.iloc[0]['total_revenue']) - 1799630.54) < 0.01, \
    (f"Q3: expected total_revenue ≈ 1,799,630.54 at score 1, got {q3.iloc[0]['total_revenue']} — "
     f"1,812,828.22 is the fanned-out figure from the demo")
assert int(q3.iloc[4]['order_count']) == 56885, \
    f"Q3: expected 56,885 orders at score 5, got {int(q3.iloc[4]['order_count']):,}"
assert abs(float(q3.iloc[4]['avg_items_per_order']) - 1.11) < 0.01, \
    f"Q3: expected avg_items_per_order ≈ 1.11 at score 5, got {q3.iloc[4]['avg_items_per_order']}"
assert abs(float(q3.iloc[4]['total_revenue']) - 7660968.12) < 0.01, \
    f"Q3: expected total_revenue ≈ 7,660,968.12 at score 5, got {q3.iloc[4]['total_revenue']}"
print("✅ Q3 correct")
q3  # show the result of your query

## Question 4 — Which categories do customers actually *like*?

Revenue tells you what sells. It does not tell you what customers are happy they bought. Put the two sides of Wednesday's session together: the three-table category chain from Concept 1, plus a review score joined on `order_id` from Concept 2 — with the fan-out lesson from Q3 applied.

Build the `rev` CTE first, exactly as the demo's "safe pattern" cell did:

```sql
WITH rev AS (
    SELECT order_id, AVG(review_score) AS review_score
    FROM order_reviews
    GROUP BY order_id
)
```

Then chain `order_items oi` → `products p` (on `product_id`) → `product_category_translation t` (on `product_category_name`), and add a **plain inner** `JOIN rev r ON oi.order_id = r.order_id`. Inner, not `LEFT` — an order nobody reviewed has no score to average, so it has no business in a satisfaction ranking. Return the **top 5 categories by average review score**, among categories with **at least 1,000 distinct orders**, with four columns:

- `category`
- `order_count` — `COUNT(DISTINCT oi.order_id)`
- `avg_review` — `ROUND(AVG(r.review_score), 2)`
- `total_revenue` — `ROUND(SUM(oi.price), 2)`

`HAVING COUNT(DISTINCT oi.order_id) >= 1000`, then `ORDER BY avg_review DESC LIMIT 5`.

**Expected:** 5 rows.

| category | order_count | avg_review | total_revenue |
|---|---|---|---|
| luggage_accessories | 1,030 | 4.32 | 139,845.20 |
| stationery | 2,295 | 4.20 | 229,478.80 |
| pet_shop | 1,701 | 4.18 | 213,246.92 |
| perfumery | 3,150 | 4.17 | 397,490.00 |
| toys | 3,853 | 4.16 | 478,878.80 |

Now read it as a business person. Not one of these five appears in the demo's top-10 *revenue* table. `luggage_accessories` leads on satisfaction at 4.32 on a mere R$139,845.20 of revenue, while `health_beauty` — the R$1.26m revenue champion — sits back at 4.14. The categories that earn the most and the categories that delight the most are two different lists, and it took a four-table query to see it.

In [ ]:
%%sql q4 <<
-- Your query here

In [ ]:
# --- CHECK Q4 — do not edit ---
for col in ['category', 'order_count', 'avg_review', 'total_revenue']:
    assert col in q4.columns, f"Q4: missing the '{col}' column — check your SELECT aliases"
assert q4.shape[0] == 5, f"Q4: expected 5 rows, got {q4.shape[0]} — did you LIMIT 5?"
assert list(q4['category']) == ['luggage_accessories', 'stationery', 'pet_shop',
                                'perfumery', 'toys'], \
    (f"Q4: unexpected category order — got {list(q4['category'])}; order by avg_review DESC "
     f"and keep HAVING COUNT(DISTINCT oi.order_id) >= 1000")
assert abs(float(q4.iloc[0]['avg_review']) - 4.32) < 0.01, \
    f"Q4: expected luggage_accessories avg_review ≈ 4.32, got {q4.iloc[0]['avg_review']}"
assert int(q4.iloc[0]['order_count']) == 1030, \
    (f"Q4: expected luggage_accessories order_count = 1,030, got "
     f"{int(q4.iloc[0]['order_count']):,} — 1,034 means you used LEFT JOIN rev instead of "
     f"a plain JOIN, which keeps unreviewed orders in the count")
assert abs(float(q4.iloc[0]['total_revenue']) - 139845.20) < 0.01, \
    f"Q4: expected luggage_accessories total_revenue ≈ 139,845.20, got {q4.iloc[0]['total_revenue']}"
assert abs(float(q4.iloc[1]['avg_review']) - 4.20) < 0.01, \
    f"Q4: expected stationery avg_review ≈ 4.20, got {q4.iloc[1]['avg_review']}"
assert int(q4.iloc[4]['order_count']) == 3853, \
    f"Q4: expected toys order_count = 3,853, got {int(q4.iloc[4]['order_count']):,}"
assert abs(float(q4.iloc[4]['avg_review']) - 4.16) < 0.01, \
    f"Q4: expected toys avg_review ≈ 4.16, got {q4.iloc[4]['avg_review']}"
print("✅ Q4 correct")
q4  # show the result of your query

## When all four are green

Look back at what you just wrote. Every one of the four queries followed the same shape: start from `order_items`, follow shared keys outward one `JOIN ... ON ...` at a time, then collapse the result with `GROUP BY`. Q1 and Q2 walked the category chain — two joins, three tables. Q4 walked it and then bolted a fourth table on. The syntax never got harder; only the reasoning about row counts did.

Three habits to carry into Thursday:

- **Alias every table and prefix every column.** `oi`, `p`, `t`, `r` — with three or four tables in play, a bare `product_category_name` is ambiguous and SQLite refuses it outright.
- **Check the grain before you trust the number.** If your `order_count` looks suspiciously round or suspiciously large, you probably counted joined rows instead of orders. Q3 showed exactly what that costs: 76 phantom orders and R$13,197.68 of phantom revenue in a single row.
- **`HAVING` filters groups; `WHERE` filters rows.** Q1's "at least 500 orders" and Q4's "at least 1,000 orders" are both conditions on an aggregate, so neither could have gone in `WHERE`.

Stuck on one? Re-read the demo section it extends — Q1 and Q2 build on Concept 1, Q3 rewrites Concept 2 safely, Q4 is the "safe pattern" cell from *Going deeper* pointed at a new ranking. And if you drafted with DeepSeek, remember that a green ✅ is the only evidence that counts.

---
**Coming up Thursday:** geographic revenue analysis — chaining `orders → customers → order_payments`, computing delivery duration from two TEXT timestamps with `julianday()`, and pushing the same join chain out to four and five tables.